 # 00_pull_clean notebook

#### This notebook takes in the QLFS data from South Africa, and the PNAD Continua data from Brazil. It transforms them into Pandas dataframes to be used in following notebooks. Data are downloaded manually(see ReadMe for details), and this notebook loads the raw files. It then cleans the data and selects the columns I want while creating a new booleen to identify if a given region is a metro area.
##### This notebook will get more complex as I continue working on this project, but for now, it is fairly simple.

Loads the raw QLFS data for South Africa (2010 Q4) and produces the dta file for later notebooks. It also loads the PNAD Brazil data and produces the csv for later notebooks.

In [1]:
# create data name for the path variable, so others can execute code (per instructions on final project)
import pandas as pd
DATA = "../data/"

## QLFS South Africa (2010)

### Pull data

In [2]:
## load data on South Africa labor stats
df=labor_data = pd.read_stata(DATA + "southafrica_qss.dta")
print(df.head())

print(df["Metro_code"].unique())
print(df["Status_Exp"].unique())

                 UQNO  PERSONNO      Province Q12NIGHTS Q13GENDER  Q14AGE  \
0  101000170000003001         1  Western Cape       Yes      Male      52   
1  101000170000003001         2  Western Cape       Yes    Female      50   
2  101000170000005101         1  Western Cape       Yes      Male      39   
3  101000170000007201         1  Western Cape       Yes      Male      40   
4  101000170000009301         1  Western Cape       Yes      Male      73   

  Q15POPULATION                       Q16MARITALSTATUS  \
0         White                                Married   
1         White                                Married   
2      Coloured  Living together like husband and wife   
3         White                   Divorce or separated   
4      Coloured  Living together like husband and wife   

                         Q17EDUCATION Q20SELFRESPOND  ... age_grp1  \
0          Grade 10/Standard 8/Form 3             No  ...    50-54   
1  Grade 12/Standard 10/Form 5/Matric           

### Clean data

In [3]:
# create list of hosting metros
host_metros = ["Cape Town", "Nelson Mandela Metro", "eThekwini", "Tshwane", "Johannesburg"]

# use lamda function to create variable on hosting group (yes or no) in dataframe
df["host_group"] = df["Metro_code"].apply(lambda x: "Metro Host" if x in host_metros else "Non-Host")

print(df["host_group"].value_counts())

host_group
Non-Host      64318
Metro Host    19039
Name: count, dtype: int64


### Subset data and save

In [4]:
# create list of variables from raw data to keep
keep_columns = ["Province", "Q13GENDER", "Q14AGE", "Q15POPULATION", "Q17EDUCATION",
             "InactReason", "At_least_1", "Infempl", "Geo_type", "Hrswrk",
             "Status_Exp", "Metro_code", "host_group"]

# make df with only the list of cols to keep
df_clean = df[keep_columns]
print(df_clean.shape)

# save back to computer/repo
df_clean.to_stata(DATA + "cleaned_southafrica_2010.dta", write_index=False)
print("Saved file")

(83357, 13)
Saved file


/var/folders/bj/gdljk_3d7b51czs3g9bn23zc0000gn/T/ipykernel_97324/3697253717.py:11: ValueLabelTypeMismatch: 
Stata value labels (pandas categories) must be strings. Column Hrswrk contains
non-string labels which will be converted to strings.  Please check that the
Stata data file created has not lost information due to duplicate labels.

  df_clean.to_stata(DATA + "cleaned_southafrica_2010.dta", write_index=False)


### PNAD Continua Brazil: 2012, 2014, 2016 (all Q4)
##### Labor force information for people over 14

#### Load data

In [11]:
## load data on Brazil labor stats
## all col names intitally in Portugese, so i translated them and made my own list to apply to df
col_names = ["level", "code", "metro",
             "total_2012", "emp_2012", "unemp_2012", "na_2012",
             "total_2014", "emp_2014", "unemp_2014", "na_2014",
             "total_2016", "emp_2016", "unemp_2016", "na_2016"]

brazil_df = pd.read_excel(DATA + "Brazil_intital.xlsx", header=None, names=col_names)

print(brazil_df.shape)
print(brazil_df["metro"])

(20, 15)
5                 Manaus (AM)
6                  Belém (PA)
7                 Macapá (AP)
8        Grande São Luís (MA)
9              Fortaleza (CE)
10                 Natal (RN)
11           João Pessoa (PB)
12                Recife (PE)
13                Maceió (AL)
14               Aracaju (SE)
15              Salvador (BA)
16        Belo Horizonte (MG)
17        Grande Vitória (ES)
18        Rio de Janeiro (RJ)
19             São Paulo (SP)
20              Curitiba (PR)
21         Florianópolis (SC)
22          Porto Alegre (RS)
23    Vale do Rio Cuiabá (MT)
24               Goiânia (GO)
Name: metro, dtype: object


#### Clean data and create host group booleen col

In [16]:
# keep only metropolitan region rows and drop SIDRA footnotes (uneeded info in datatable)
brazil_df = brazil_df[brazil_df["level"] == "RM"]

# create list of hosting metros
brazil_hosts = ["Rio de Janeiro (RJ)", "São Paulo (SP)", "Fortaleza (CE)"]

# same logic here as South Africa
brazil_df["host_group"] = brazil_df["metro"].apply(
    lambda x: "Metro Host" if x in brazil_hosts else "Non-Host")

print(brazil_df["host_group"].value_counts())

host_group
Non-Host      17
Metro Host     3
Name: count, dtype: int64


#### Save data
No subset here because when selecting data from website, I got to pick the variables I wanted.

In [18]:
# save data as csv
brazil_df.to_csv(DATA + "cleaned_brazil_pnad.csv", index=False)
print("Saved file")

Saved file
